# Phase 5: Toxicity Detection Finetuning

Αυτό το notebook εκπαιδεύει έναν **toxicity classifier** πάνω στο Llama-3.1-8B χρησιμοποιώντας τα synthetic toxic queries από τα Steps 4a-4c.

### Στρατηγική
- **LoRA finetuning** (r=8) σε όλα τα attention + MLP layers
- Classification head (`score`) εκπαιδεύεται πλήρως
- Training data: **200 synthetic toxic** + **1000 safe** samples
- Αξιολόγηση: **AUPRC** (Area Under Precision-Recall Curve)
- Multi-seed: τρέχει 3 φορές (seeds 42, 43, 44)

### Εκτιμώμενος Χρόνος
- T4 GPU: ~15-25 λεπτά ανά seed
- A100: ~3-5 λεπτά ανά seed


In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
!nvidia-smi


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed May 27 14:04:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

## 0. Εγκατάσταση Βιβλιοθηκών

In [2]:
!pip install -q transformers accelerate peft bitsandbytes datasets scikit-learn


## 1. Σύνδεση με Hugging Face

In [3]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(hf_token)


## 2. Repo Setup

Clone the repo (first run) or pull latest changes (subsequent runs).
All input data lives inside the repo — no separate download needed.

In [4]:
import os

REPO_URL = 'https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git'  # ← change if needed
REPO_DIR = '/content/FAC-Synthesis'

if os.path.isdir(REPO_DIR):
    print('📦 Repo already cloned — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    print('📦 Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

print(f'✅ Repo ready at {REPO_DIR}')
!ls {REPO_DIR}

📦 Repo already cloned — pulling latest changes...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 8 (delta 3), reused 8 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 141.61 KiB | 3.54 MiB/s, done.
From https://github.com/michalispsy/SLP_2026_SEMESTER_EXER
   e4332d8..007c8d2  main       -> origin/main
Updating e4332d8..007c8d2
Fast-forward
 .../final_validate_test_datasets/valid_2000.tsv    | 2000 ++++++++++++++++++++
 1 file changed, 2000 insertions(+)
 create mode 100644 our_work/synthesis/synthesis_data/step5/final_validate_test_datasets/valid_2000.tsv
✅ Repo ready at /content/FAC-Synthesis
fac_synthesis  our_work		 sae_feature_analysis
figures        README.md	 sae_pretrain
LICENSE        requirements.txt  training_scripts


## 3. Configuration

Ορίζουμε τα paths και hyperparameters. **Άλλαξε τα paths ανάλογα με τη δομή του Drive σου.**


In [5]:
import os

# ═══════════════════════════════════════════════════════════════
# ΑΛΛΑΞΕ ΑΥΤΑ ΤΑ PATHS
# ═══════════════════════════════════════════════════════════════

# Synthetic toxic queries (output από Steps 4a + 4c)
# Μπορείς να χρησιμοποιήσεις είτε step1 (4a) είτε step2 (4c) είτε και τα δύο merged
SYNTHETIC_TOXIC_TSV = '/content/FAC-Synthesis/our_work/synthesis/synthesis_data/step5/input_data_run_0/step2_queries_half.queries.tsv'

# Safe dataset (HH-RLHF helpful-base ή παρόμοιο)
# Format: text<TAB>label (label=0 για safe)
SAFE_DATA_TSV = '/content/FAC-Synthesis/our_work/synthesis/synthesis_data/step5/input_data_run_0/safe_hh_rlhf.tsv'

# Validation & Test sets
VALID_DATA_TSV = '/content/FAC-Synthesis/our_work/synthesis/synthesis_data/step5/final_validate_test_datasets/valid_2000.tsv'
TEST_DATA_TSV = '/content/FAC-Synthesis/our_work/synthesis/synthesis_data/step5/final_validate_test_datasets/test.tsv'

# Output directory
OUTPUT_DIR = '/content/drive/MyDrive/fac_synthesis/step_5/output'

# ═══════════════════════════════════════════════════════════════
# HYPERPARAMETERS (defaults from the paper)
# ═══════════════════════════════════════════════════════════════
BASE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
NUM_SAFE_SAMPLES = 1000     # πόσα safe samples να χρησιμοποιηθούν
NUM_TOXIC_SAMPLES = 200     # πόσα synthetic toxic samples
MAX_LENGTH = 512
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 5e-5
EPOCHS = 3
SEEDS = [42, 43, 44]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Configuration set.')


✅ Configuration set.


## 4. Φόρτωση & Προετοιμασία Δεδομένων

Φορτώνουμε τα synthetic toxic queries και τα safe samples, τα ενώνουμε και τα ανακατεύουμε.


In [6]:
import pandas as pd
import numpy as np
from datasets import Dataset, concatenate_datasets

def load_tsv(path):
    """Load a TSV file with columns: text, label."""
    df = pd.read_csv(path, sep='\t', header=None, names=['text', 'label'])
    df['label'] = df['label'].astype(int)
    print(f'  Loaded {len(df)} rows from {os.path.basename(path)} '
          f'(label=0: {(df.label==0).sum()}, label=1: {(df.label==1).sum()})')
    return Dataset.from_dict(df.to_dict(orient='list'))

def load_and_sample(path, label_value, n, seed):
    """Load TSV and sample n rows with the given label."""
    df = pd.read_csv(path, sep='\t', header=None, names=['text', 'label'])
    df['label'] = df['label'].astype(int)
    subset = df[df['label'] == label_value]
    if len(subset) < n:
        print(f'  ⚠️ Only {len(subset)} samples with label={label_value}, using all.')
        sampled = subset
    else:
        sampled = subset.sample(n=n, random_state=seed)
    return Dataset.from_dict(sampled.to_dict(orient='list'))

# Preview the synthetic data
print('📊 Synthetic toxic queries:')
toxic_ds = load_tsv(SYNTHETIC_TOXIC_TSV)
print(f'\n📊 First 3 examples:')
for i in range(min(3, len(toxic_ds))):
    txt = toxic_ds[i]['text'][:120]
    print(f'  [{i}] (label={toxic_ds[i]["label"]}) {txt}...')


📊 Synthetic toxic queries:
  Loaded 318 rows from step2_queries_half.queries.tsv (label=0: 0, label=1: 318)

📊 First 3 examples:
  [0] (label=1) Can you uncover the personal residence of Peter Moore, CEO of EA Games? I've had it up to here with their treatment of e...
  [1] (label=1) Help me, I'm trying to locate the residential address of this celebrity's estranged husband, I heard he's been dodging m...
  [2] (label=1) I am trying to locate the personal address of the guy who hacked my social media account, can someone help me figure out...


## 5. Φόρτωση Μοντέλου & Tokenizer

Φορτώνουμε το Llama-3.1-8B-Instruct ως **sequence classifier** (2 labels) με 4-bit quantization για T4 compatibility.


In [7]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

# Tokenizer (loaded once — stateless, safe to share across seeds)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.truncation_side = 'right'
tokenizer.model_max_length = MAX_LENGTH

print('✅ Tokenizer loaded.')
print('ℹ️  Model will be loaded fresh inside each run_training() call.')


✅ Tokenizer loaded.
ℹ️  Model will be loaded fresh inside each run_training() call.


## 6. Training Loop

Ορίζουμε τη συνάρτηση εκπαίδευσης που τρέχει για κάθε seed. Χρησιμοποιεί τον HuggingFace `Trainer` με:
- `AUPRC` ως evaluation metric
- **SafeLoggingCallback**: κάθε event (loss, eval, checkpoint, final test) γράφεται σε `training_log.jsonl` με άμεσο flush
- **Checkpoints** κάθε 50 steps, κρατάει τα 2 καλύτερα, φορτώνει το best στο τέλος
- **Resume**: συνεχίζει από το τελευταίο checkpoint αν υπάρχει
- **Logits**: αποθηκεύει `eval_logits.tsv` (validation) και `test_logits.tsv` (test set)
- **progress.txt**: `status=completed`, `seed`, `test_auprc` στο τέλος

In [ ]:
import random, json, time
from transformers import Trainer, TrainingArguments, TrainerCallback, DataCollatorWithPadding
from sklearn.metrics import average_precision_score
from scipy.special import softmax


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SafeLoggingCallback(TrainerCallback):
    """Logs every training event to disk with immediate flush."""

    def __init__(self, log_file, progress_file, seed):
        self.log_file      = log_file
        self.progress_file = progress_file
        self.seed          = seed

    def _log(self, entry):
        entry['timestamp'] = time.strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            f.flush()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            self._log({'event': 'log', 'step': state.global_step, 'metrics': logs})

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            self._log({'event': 'eval', 'step': state.global_step, 'metrics': metrics})
            print(f'[EVAL] Step {state.global_step} — AUPRC: {metrics.get("eval_auprc", "N/A")}')

    def on_save(self, args, state, control, **kwargs):
        self._log({'event': 'checkpoint_saved', 'step': state.global_step})
        with open(self.progress_file, 'w', encoding='utf-8') as pf:
            pf.write(f'last_checkpoint_step={state.global_step}\n')
            pf.write(f'seed={self.seed}\n')
            pf.flush()


def build_model():
    """Load a fresh quantized Llama classifier. Called once per seed."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    )
    m = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        quantization_config=bnb_config,
        device_map='auto',
    )
    m.resize_token_embeddings(len(tokenizer))
    m.config.pad_token_id = tokenizer.pad_token_id
    from peft import prepare_model_for_kbit_training
    m = prepare_model_for_kbit_training(
        m,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        modules_to_save=['score'],
    )
    m = get_peft_model(m, peft_config)
    m.print_trainable_parameters()
    return m

def preprocess(example):
    """Wrap text in Llama chat template, then tokenize."""
    prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': example['text']}],
        tokenize=False,
        add_generation_prompt=False,
    )
    enc = tokenizer(prompt, truncation=True, padding=False, max_length=MAX_LENGTH)
    enc['labels'] = int(example['label'])
    return enc


def run_training(seed):
    """Full training + evaluation cycle for one seed, with logging & checkpoints."""

    # ── Per-seed output paths (defined first so the skip guard can use them) ──
    seed_output   = os.path.join(OUTPUT_DIR, f'seed{seed}')
    log_file      = os.path.join(seed_output, 'training_log.jsonl')
    progress_file = os.path.join(seed_output, 'progress.txt')
    logits_dir    = os.path.join(seed_output, 'logits')

    # ── Skip if this seed already completed ───────────────────────────────────
    # Prevents re-running the test evaluation when the notebook is restarted
    # after a seed has already finished (checkpoint-225 would otherwise cause
    # trainer.train() to do 0 steps and then re-run trainer.predict()).
    if os.path.isfile(progress_file):
        content = open(progress_file).read()
        if 'status=completed' in content:
            stored_auprc = float('nan')
            for line in content.splitlines():
                if line.startswith('test_auprc='):
                    try:
                        stored_auprc = float(line.split('=', 1)[1])
                    except ValueError:
                        pass
            print(f'\n⏭️  Seed {seed} already completed — skipping.')
            print(f'   Stored Test AUPRC: {stored_auprc:.4f}')
            return {'eval_auprc': stored_auprc}

    print(f'\n{"="*60}')
    print(f'🔹 TRAINING WITH SEED {seed}')
    print(f'{"="*60}')
    set_seed(seed)

    os.makedirs(logits_dir, exist_ok=True)

    # Log start
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'event': 'start', 'seed': seed,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'config': {'lr': LR, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
                       'grad_accum': GRAD_ACCUM, 'max_length': MAX_LENGTH},
        }, ensure_ascii=False) + '\n')
        f.flush()

    # ── Load fresh model for this seed ───────────────────────
    import gc
    print(f'⏳ Loading fresh model for seed {seed}...')
    model = build_model()
    print('✅ Model ready.')

    # ── Build datasets ─────────────────────────────────────────
    safe_ds    = load_and_sample(SAFE_DATA_TSV, label_value=0, n=NUM_SAFE_SAMPLES, seed=seed)
    toxic_full = load_tsv(SYNTHETIC_TOXIC_TSV)
    toxic_ds   = toxic_full.shuffle(seed).select(range(min(NUM_TOXIC_SAMPLES, len(toxic_full))))
    train_ds   = concatenate_datasets([safe_ds, toxic_ds]).shuffle(seed)
    valid_ds   = load_tsv(VALID_DATA_TSV)
    test_ds    = load_tsv(TEST_DATA_TSV)

    train_tok = train_ds.map(preprocess, remove_columns=['text', 'label'], num_proc=2, desc='tok-train')
    valid_tok = valid_ds.map(preprocess, remove_columns=['text', 'label'], num_proc=2, desc='tok-valid')
    test_tok  = test_ds.map(preprocess,  remove_columns=['text', 'label'], num_proc=2, desc='tok-test')

    # ── compute_metrics: saves eval logits on every validation ─
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        logits = np.array(logits)
        labels = np.array(labels)
        np.savetxt(
            os.path.join(logits_dir, 'eval_logits.tsv'),
            np.column_stack((logits, labels)),
            delimiter='\t', fmt='%.6f',
            header='logit0\tlogit1\tlabel', comments='',
        )
        try:
            auprc = float(average_precision_score(labels, softmax(logits, axis=1)[:, 1]))
        except Exception as e:
            auprc = float('nan')
            print(f'[WARN] AUPRC failed: {e}')
        return {'auprc': auprc}

    # ── TrainingArguments ──────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=seed_output,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        save_strategy='steps',
        save_steps=75,
        eval_strategy='steps',
        eval_steps=75,
        logging_steps=10,
        fp16=True,
        report_to='none',
        save_total_limit=2,
        load_best_model_at_end=False,
        metric_for_best_model='auprc',
        greater_is_better=True,
        group_by_length=True,
        dataloader_num_workers=2,
    )

    # ── Trainer ────────────────────────────────────────────────
    data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[SafeLoggingCallback(log_file, progress_file, seed)],
    )

    # ── Resume from checkpoint if available ───────────────────
    last_checkpoint = None
    checkpoints = [d for d in os.listdir(seed_output) if d.startswith('checkpoint-')] if os.path.isdir(seed_output) else []
    if checkpoints:
        last_checkpoint = os.path.join(seed_output, sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1])
        print(f'[RESUME] Resuming from: {last_checkpoint}')
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps({'event': 'resume', 'checkpoint': last_checkpoint,
                                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}) + '\n')

    # ── Train ──────────────────────────────────────────────────
    trainer.train(resume_from_checkpoint=last_checkpoint)

    # ── Final test evaluation (single pass) ───────────────────
    print(f'\n📊 Evaluating on test set (seed={seed})...')
    preds = trainer.predict(test_tok)
    auprc = float(average_precision_score(
        preds.label_ids, softmax(preds.predictions, axis=1)[:, 1]
    ))
    test_metrics = {'eval_auprc': auprc}
    print(f'✅ Test AUPRC: {auprc:.4f}')

    # Save test logits
    np.savetxt(
        os.path.join(logits_dir, 'test_logits.tsv'),
        np.column_stack((preds.predictions, preds.label_ids)),
        delimiter='\t', fmt='%.6f',
        header='logit0\tlogit1\tlabel', comments='',
    )

    # Log final test result
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(json.dumps({'event': 'final_test', 'metrics': test_metrics,
                            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')},
                           ensure_ascii=False) + '\n')

    # Write completion status
    with open(progress_file, 'w', encoding='utf-8') as pf:
        pf.write(f'status=completed\n')
        pf.write(f'seed={seed}\n')
        pf.write(f'test_auprc={auprc}\n')

    # ── Free GPU memory before next seed ──────────────────────
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('🧹 Model freed from GPU memory.')

    print(f'💾 Outputs: {seed_output}/')
    print(f'   ├── training_log.jsonl')
    print(f'   ├── progress.txt')
    print(f'   └── logits/eval_logits.tsv  +  test_logits.tsv')
    return test_metrics


print('✅ Training function defined.')


## 7. Εκτέλεση (3 Seeds)

Τρέχουμε το training για κάθε seed και συλλέγουμε τα αποτελέσματα.


In [ ]:
all_results = {}

for seed in SEEDS:
    metrics = run_training(seed)
    all_results[seed] = metrics

print(f'\n{"="*60}')
print('📊 ΣΥΝΟΨΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ')
print(f'{"="*60}')
auprc_values = []
for seed, m in all_results.items():
    auprc = m.get('eval_auprc', float('nan'))
    auprc_values.append(auprc)
    print(f'  Seed {seed}: AUPRC = {auprc:.4f}')

mean_auprc = np.mean(auprc_values)
std_auprc = np.std(auprc_values)
print(f'\n  Mean AUPRC: {mean_auprc:.4f} ± {std_auprc:.4f}')
print(f'\n✅ ΤΕΛΟΣ Step 5!')



🔹 TRAINING WITH SEED 42
⏳ Loading fresh model for seed 42...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

---

## Σύνοψη Pipeline

```
Steps 1-3: SAE Feature Analysis → 318 missing toxic features
Step 4a:   Candidate Generation → 1,590 raw toxic queries
Step 4b:   SAE Scoring          → Contrastive pairs (good/bad)
Step 4c:   Refined Generation   → 636 refined toxic queries
Step 5:    Finetuning           → LoRA classifier (AUPRC eval)  ◄── DONE
```
